# Predictive Maintenance for Sterilization Equipment

**End-to-end ML pipeline running entirely in Snowflake** — from raw cycle telemetry to registered production model.

**What this notebook demonstrates:**
1. Connect to pre-loaded IIoT cycle data (47K cycles, 69.6M analog readings, 113 devices)
2. Explore engineered features derived from 6-channel sterilizer sensor data
3. Train an XGBoost classifier to predict cycle pass/fail
4. Evaluate model performance
5. Register the model in Snowflake Model Registry for production inference

**Infrastructure:** Standard MEDIUM warehouse (4 credits/hr) — no specialized compute required.

---
## 1. Setup and Connection

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, when, lit, count, avg, round as sf_round
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.registry import Registry

# Connect
session = Session.builder.config("connection_name", "akelkar-demo-account").create()
session.sql("USE ROLE SF_INTELLIGENCE_DEMO").collect()
session.sql("USE DATABASE IIOT_PREDICTIVE_MAINT_DB").collect()
session.sql("USE WAREHOUSE IIOT_FEATURE_WH").collect()

print(f"Connected to: {session.get_current_account()}")
print(f"Warehouse:    {session.get_current_warehouse()} (Standard MEDIUM, 4 cr/hr)")
print(f"Database:     {session.get_current_database()}")

---
## 2. Data Overview

We've already ingested **47,000 sterilizer cycle XMLs** and engineered features from 69.6M analog time-series readings. Each cycle has 6 sensor channels sampled at 5-second intervals across 4 phases:

| Phase | Duration | Key Signals |
|-------|----------|-------------|
| Conditioning | 4-6 min | Temperature ramp, pressure rise |
| Sterilize | 4-18 min | Plateau at 134C / 210 kPa |
| Exhaust | 1.5-2.5 min | Rapid pressure drop |
| Drying | 4-6 min | Vacuum, temperature settling |

In [ ]:
# Data volumes
print("=== Data Volumes ===")
tables = [
    ("RAW.CYCLE", "Cycle summary (1 row per sterilization cycle)"),
    ("RAW.CYCLE_ANALOG", "Analog time-series (6 channels x ~230 readings/cycle)"),
    ("RAW.DEVICE", "Device master (sterilizer registry)"),
    ("RAW.ALARM", "Alarm events (failed cycles)"),
    ("FEATURES.CYCLE_FEATURES", "Engineered ML features"),
]
for table, desc in tables:
    ct = session.table(table).count()
    print(f"  {table:30s} {ct:>12,}  — {desc}")

---
## 3. Feature Exploration

Each cycle produces **26 engineered features** from the raw analog readings — statistical aggregates per sensor channel that capture equipment health signals.

In [ ]:
# Load feature table
df = session.table("FEATURES.CYCLE_FEATURES")

# Failure rate breakdown
print("=== Cycle Outcome Distribution ===")
outcome_df = df.group_by("SUMMARY_CYCLE_STATUS").agg(count("*").alias("COUNT"))
outcome_df.sort("SUMMARY_CYCLE_STATUS").show()

failure_labels = {0: "Pass", 1: "Seal Leak", 2: "Temp Drift", 3: "Pressure Anomaly", 4: "Sensor Fault"}
print("\nFailure mode key:")
for k, v in failure_labels.items():
    print(f"  Status {k} = {v}")

In [ ]:
# Compare key features between passing and failing cycles
print("=== Feature Comparison: Pass vs. Fail ===\n")

pass_df = df.filter(col("SUMMARY_CYCLE_STATUS") == 0)
fail_df = df.filter(col("SUMMARY_CYCLE_STATUS") != 0)

compare_cols = ["LEAK_RATE", "STDDEV_CHAMBER_PRESSURE", "MAX_CHAMBER_TEMP", "TEMP_PROBE_DIVERGENCE"]
print(f"{'Feature':<28} {'Pass (avg)':>12} {'Fail (avg)':>12} {'Signal':>10}")
print("-" * 65)

for c in compare_cols:
    pass_avg = pass_df.select(avg(col(c))).collect()[0][0]
    fail_avg = fail_df.select(avg(col(c))).collect()[0][0]
    signal = ">>>" if abs(fail_avg - pass_avg) / max(abs(pass_avg), 0.001) > 0.2 else ""
    print(f"  {c:<26} {pass_avg:>12.3f} {fail_avg:>12.3f} {signal:>10}")

---
## 4. Model Training

Training an **XGBoost gradient-boosted classifier** to predict cycle pass/fail.

- Binary target: `IS_FAILURE` (0 = pass, 1 = any failure mode)
- 26 numeric features derived from 6 sensor channels
- 80/20 train/test split
- All computation happens **inside Snowflake** — data never leaves the platform

In [ ]:
# Define features
FEATURE_COLS = [
    "LEAK_RATE", "MIN_TEMP", "MAX_TEMP", "TEMP_RANGE", "CYCLE_COUNT",
    "AVG_CHAMBER_PRESSURE", "MAX_CHAMBER_PRESSURE", "MIN_CHAMBER_PRESSURE", "STDDEV_CHAMBER_PRESSURE",
    "AVG_JACKET_PRESSURE", "MAX_JACKET_PRESSURE", "STDDEV_JACKET_PRESSURE",
    "AVG_CHAMBER_TEMP", "MAX_CHAMBER_TEMP", "MIN_CHAMBER_TEMP", "STDDEV_CHAMBER_TEMP",
    "AVG_CHAMBER_TEMP_2", "TEMP_PROBE_DIVERGENCE",
    "AVG_DRAIN_TEMP", "MAX_DRAIN_TEMP",
    "AVG_JACKET_TEMP", "MAX_JACKET_TEMP",
    "AVG_PRESSURE_DIFFERENTIAL",
    "TOTAL_READINGS", "PRESSURE_NONZERO_COUNT", "TEMP_NONZERO_COUNT",
]

# Create binary target
df_labeled = df.with_column("IS_FAILURE", when(col("SUMMARY_CYCLE_STATUS") != 0, lit(1)).otherwise(lit(0)))

# Split
train_df, test_df = df_labeled.random_split([0.8, 0.2], seed=42)
print(f"Training set: {train_df.count():,} cycles")
print(f"Test set:     {test_df.count():,} cycles")
print(f"Features:     {len(FEATURE_COLS)}")

In [ ]:
# Train
print("Training XGBoost classifier...")
start = time.time()

clf = XGBClassifier(
    input_cols=FEATURE_COLS,
    label_cols=["IS_FAILURE"],
    output_cols=["PREDICTED_FAILURE"],
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
)
clf.fit(train_df)

elapsed = time.time() - start
credits_used = elapsed / 3600 * 4  # MEDIUM = 4 cr/hr

print(f"\nTraining complete.")
print(f"  Wall time:   {elapsed:.1f} seconds")
print(f"  Credits:     {credits_used:.4f} (at 4 cr/hr)")
print(f"  Cost:        ${credits_used * 3:.4f} (at $3/credit on-demand)")

---
## 5. Model Evaluation

In [ ]:
# Predict on test set
predictions = clf.predict(test_df)

# Overall accuracy
total = predictions.count()
correct = predictions.filter(col("PREDICTED_FAILURE") == col("IS_FAILURE")).count()
accuracy = correct / total

# Failure detection metrics
true_pos = predictions.filter((col("PREDICTED_FAILURE") == 1) & (col("IS_FAILURE") == 1)).count()
false_pos = predictions.filter((col("PREDICTED_FAILURE") == 1) & (col("IS_FAILURE") == 0)).count()
false_neg = predictions.filter((col("PREDICTED_FAILURE") == 0) & (col("IS_FAILURE") == 1)).count()
true_neg = predictions.filter((col("PREDICTED_FAILURE") == 0) & (col("IS_FAILURE") == 0)).count()

precision = true_pos / max(true_pos + false_pos, 1)
recall = true_pos / max(true_pos + false_neg, 1)
f1 = 2 * precision * recall / max(precision + recall, 0.001)

print("=== Model Performance ===\n")
print(f"  Accuracy:   {accuracy:.4f}")
print(f"  Precision:  {precision:.4f}  (of predicted failures, how many were real)")
print(f"  Recall:     {recall:.4f}  (of actual failures, how many did we catch)")
print(f"  F1 Score:   {f1:.4f}")
print(f"\n  Confusion Matrix:")
print(f"                    Predicted Pass  Predicted Fail")
print(f"    Actual Pass     {true_neg:>10,}     {false_pos:>10,}")
print(f"    Actual Fail     {false_neg:>10,}     {true_pos:>10,}")

---
## 6. Register in Model Registry

The trained model is registered in Snowflake's Model Registry — making it available for:
- **Batch inference** via SQL (`SELECT model!PREDICT(...)`)
- **Version management** (A/B testing, rollback)
- **Governance** (lineage, audit trail)
- **No deployment infrastructure needed** — it's already in the platform

In [ ]:
# Register model
reg = Registry(session=session, database_name="IIOT_PREDICTIVE_MAINT_DB", schema_name="ML")

model_version = reg.log_model(
    model=clf,
    model_name="CYCLE_FAILURE_CLASSIFIER",
    version_name="v1",
    comment=f"XGBoost binary classifier | Accuracy: {accuracy:.4f} | F1: {f1:.4f}",
    sample_input_data=train_df.select(FEATURE_COLS).limit(10),
)

print(f"Model registered successfully.")
print(f"  Name:    {model_version.model_name}")
print(f"  Version: {model_version.version_name}")
print(f"  Location: IIOT_PREDICTIVE_MAINT_DB.ML.CYCLE_FAILURE_CLASSIFIER")

---
## 7. Production Inference Demo

Show how the registered model scores new cycles — this is what runs daily in production.

In [ ]:
# Simulate scoring today's cycles (grab a random sample as "new" cycles)
new_cycles = df_labeled.sample(n=500)  # Simulate 500 new cycles arriving today

# Score using the registered model
model_ref = reg.get_model("CYCLE_FAILURE_CLASSIFIER").version("v1")
scored = model_ref.run(new_cycles.select(FEATURE_COLS + ["CYCLE_ID", "DEVICE_ID", "IS_FAILURE"]), function_name="predict")

# Show results
flagged = scored.filter(col("PREDICTED_FAILURE") == 1)
flagged_count = flagged.count()
total_scored = scored.count()

print(f"=== Daily Inference Results ===")
print(f"  Cycles scored:     {total_scored}")
print(f"  Predicted failures: {flagged_count} ({flagged_count/total_scored*100:.1f}%)")
print(f"\nTop flagged devices:")
flagged.group_by("DEVICE_ID").agg(count("*").alias("FLAGGED_CYCLES")).sort(col("FLAGGED_CYCLES").desc()).limit(5).show()

---
## 8. Cost Summary

All of the above — data loading, feature engineering, model training, registration, and inference — ran on a **standard MEDIUM warehouse** (4 credits/hour).

| Operation | Duration | Credits | Cost ($3/cr) |
|-----------|----------|---------|-------------|
| Feature engineering (69.6M rows) | ~5 sec | 0.005 | $0.02 |
| Model training (47K rows x 26 features) | ~30 sec | 0.03 | $0.10 |
| Daily inference (500 cycles) | < 1 sec | 0.001 | $0.003 |

**Monthly production estimate (300 devices, ~2,600 cycles/day):**
- Ingestion + features + inference: **$32-48/month**
- Weekly model retrain: **$0.40/month**
- No separate ML infrastructure. No shadow IT. One platform.

In [ ]:
# Clean up session
session.close()
print("Session closed.")